# TACK training (compact recipe)
Train only Bridge and the nudity head; SD and CLIP remain frozen. Outputs go to `outputs/training/`.

In [ ]:
from pathlib import Path
import hashlib, math
import torch
import torch.nn.functional as F
from torch import nn
from transformers import CLIPVisionModelWithProjection
from model import Bridge, ConceptHead

cache_path = Path("data/training_cache.pt")
clip_model = "openai/clip-vit-large-patch14"
model_dir = Path("models/tack-clip-vit-large-patch14")
concept = "nudity"
output = Path("outputs/training") / model_dir.name
seed, bridge_epochs, head_epochs = 7, 96, 200
times = [701, 601, 501, 401]
batch_size, microbatch, warmup = 32, 8, 100

## 1. Load frozen targets and initialize Bridge

In [ ]:
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
data = torch.load(cache_path, weights_only=True, map_location="cpu", mmap=True)
labels = data["labels"].float()
assert len(set(data["sample_keys"])) == len(labels) == len(data["split"])
ids = {s: torch.tensor([i for i, split in enumerate(data["split"]) if split == s])
       for s in ("train", "validation", "calibration")}
assert all(len(v) and set(labels[v].tolist()) == {0., 1.} for v in ids.values())
assert set(data["features"]) == set(times)
assert data["target_cls"].shape == (len(labels), 768)
assert data["target_tokens"].shape == (len(labels), 256, 768)
teacher = CLIPVisionModelWithProjection.from_pretrained(
    model_dir / "clip", torch_dtype=torch.float32,
    local_files_only=True, use_safetensors=True,
)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
bridge = Bridge(teacher).cuda()
del teacher
parameters = [p for p in bridge.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(parameters, lr=4e-4, weight_decay=0.)
order_rng = torch.Generator().manual_seed(seed)
output.mkdir(parents=True, exist_ok=True)

def features(index, timestep):
    return data["features"][timestep][index].cuda()

def alignment(index, inputs):
    cls, patches = bridge(inputs, return_tokens=True)
    return (F.mse_loss(cls, data["target_cls"][index].cuda().float())
            + F.mse_loss(patches, data["target_tokens"][index].cuda().float()))

def adapter_state():
    return {n: p.detach().cpu().clone() for n, p in bridge.named_parameters() if p.requires_grad}

## 2. Bridge: CLS + patch MSE, validation selection

In [ ]:
best_loss, best_adapter, updates = float("inf"), None, 0
total_updates = bridge_epochs * math.ceil(len(ids["train"]) / batch_size)
for epoch in range(bridge_epochs + 1):
    if epoch:
        bridge.train()
        bridge.vision.eval()
        order = ids["train"][torch.randperm(len(ids["train"]), generator=order_rng)]
        selected_times = torch.randint(len(times), (len(order),), generator=order_rng)
        for selected, ts in zip(order.split(batch_size), selected_times.split(batch_size)):
            optimizer.zero_grad(set_to_none=True)
            lr = 4e-4 * min((updates + 1) / warmup, 1.) * .5 * (1 + math.cos(math.pi * updates / total_updates))
            for group in optimizer.param_groups:
                group["lr"] = lr
            for index, t_ids in zip(selected.split(microbatch), ts.split(microbatch)):
                inputs = torch.stack([data["features"][times[int(t)]][int(i)]
                          for i, t in zip(index, t_ids)]).cuda()
                loss = alignment(index, inputs)
                assert torch.isfinite(loss)
                (loss * len(index) / len(selected)).backward()
            nn.utils.clip_grad_norm_(parameters, 1., error_if_nonfinite=True)
            optimizer.step()
            updates += 1
    bridge.eval()
    with torch.no_grad():
        valid_loss = sum(float(alignment(v, features(v, 601))) * len(v)
                         for v in ids["validation"].split(16)) / len(ids["validation"])
    if valid_loss < best_loss:
        best_loss, best_adapter = valid_loss, adapter_state()
    print(f"Bridge {epoch}: validation loss={valid_loss:.6f}")
bridge.load_state_dict({**bridge.state_dict(), **best_adapter})
bridge.eval().requires_grad_(False)

## 3. Head: balanced BCE and validation selection

In [ ]:
@torch.no_grad()
def vectors(timestep):
    return torch.cat([bridge(features(v, timestep)).cpu()
                      for v in torch.arange(len(labels)).split(16)])

embeddings = vectors(601)
mean = embeddings[ids["train"]].mean(0)
scale = embeddings[ids["train"]].std(0, unbiased=False).clamp_min(.01)
x, y = ((embeddings - mean) / scale).cuda(), labels.cuda()
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
head = ConceptHead().cuda()
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
train, valid, cal = (ids[s] for s in ("train", "validation", "calibration"))
weights = torch.where(y[train] == 1, (y[train] == 0).sum() / y[train].sum(), 1.)
best_bce, best_head = float("inf"), None
for epoch in range(1, head_epochs + 1):
    head.train()
    optimizer.zero_grad(set_to_none=True)
    loss = F.binary_cross_entropy_with_logits(head(x[train]), y[train], weight=weights)
    loss.backward()
    nn.utils.clip_grad_norm_(head.parameters(), 1., error_if_nonfinite=True)
    optimizer.step()
    if epoch in {25, 50, 100, head_epochs}:
        head.eval()
        with torch.no_grad():
            valid_weights = torch.where(y[valid] == 1, (y[valid] == 0).sum() / y[valid].sum(), 1.)
            bce = float(F.binary_cross_entropy_with_logits(head(x[valid]), y[valid], weight=valid_weights))
        if bce < best_bce:
            best_bce = bce
            best_head = {n: p.detach().cpu().clone() for n, p in head.state_dict().items()}
head.load_state_dict(best_head)
head.eval().requires_grad_(False)

## 4. Calibrate per-time thresholds and save

In [ ]:
thresholds = {}
for t in times:
    with torch.no_grad():
        scores = head(((vectors(t) - mean) / scale).cuda()).cpu()
    negatives = scores[cal][labels[cal] == 0].sort().values
    boundary = negatives[math.ceil((1 - .024) * len(negatives)) - 1]
    thresholds[t] = float(torch.nextafter(boundary, torch.tensor(float("inf"))))
bridge_path = output / Bridge.filename
torch.save(dict(state_dict=best_adapter, layer=Bridge.layer, clip_model=clip_model), bridge_path)
torch.save(dict(state_dict=best_head, normalization_mean=mean, normalization_scale=scale,
                input_schema="bridge_cls_768", concept=concept, clip_model=clip_model,
                bridge_sha256=hashlib.sha256(bridge_path.read_bytes()).hexdigest(),
                thresholds=thresholds), output / f"{concept}_head.pt")
print("Saved", bridge_path.name, "and", f"{concept}_head.pt", "to", output)
